# Stage 3 — anti-overfitting in-house adaptation

Stage 3 starts from the completed Stage 2 v2 checkpoint `best_model_stage2_v2.weights.h5`. It ranks all 80 in-house images with that checkpoint first, then lets the user define a validated 10/60 or 20/60 experiment. The training model is separate: `train()` performs its one strict checkpoint load.

**Selection warning.** Choosing only the worst-ranked images for training and the easiest images for validation would bias the result. Prefer difficulty-stratified selection across the ranking. Any split informed by this ranking is model-guided and its validation result is **not an independent test evaluation**; the 60 images are used for selection and model control.


In [ ]:
import os
import sys
from pathlib import Path


def find_repo_root():
    starts = [Path.cwd().resolve(), Path('/kaggle/working').resolve()]
    for start in starts:
        for candidate in (start, *start.parents):
            if (candidate / 'src' / 'config.py').is_file():
                return candidate
    for search_root in (Path('/kaggle/working'), Path('/kaggle/input')):
        if search_root.is_dir():
            matches = sorted(search_root.rglob('src/config.py'))
            if matches:
                return matches[0].parent.parent
    raise FileNotFoundError('Could not locate repository root containing src/config.py')


def resolve_stage2_checkpoint(repo_root):
    env_path = os.getenv('CEPHALO_STAGE2_WEIGHTS')
    candidates = [Path(env_path).expanduser()] if env_path else []
    candidates += [
        repo_root / 'checkpoints' / 'best_model_stage2_v2.weights.h5',
        Path('/kaggle/working/best_model_stage2_v2.weights.h5'),
    ]
    for search_root in (Path('/kaggle/input'), Path('/kaggle/working')):
        if search_root.is_dir():
            candidates.extend(sorted(search_root.rglob('best_model_stage2_v2.weights.h5')))
    seen = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate not in seen and candidate.is_file():
            return str(candidate)
        seen.add(candidate)
    raise FileNotFoundError(
        'best_model_stage2_v2.weights.h5 was not found. Set CEPHALO_STAGE2_WEIGHTS '
        'or attach/copy the completed Stage 2 v2 checkpoint.'
    )

REPO_ROOT = find_repo_root()
SRC_DIR = REPO_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from config import make_stage3_finetune_config, resolve_data_dir
from reproducibility import set_global_seed

FT_ROOT = os.getenv('CEPHALO_FT_ROOT', '/kaggle/input/fine-tuning-dataset-final') if Path('/kaggle').is_dir() else str(REPO_ROOT / 'data' / 'inhouse')
STAGE2_WEIGHTS = resolve_stage2_checkpoint(REPO_ROOT)
RESULTS_DIR = Path('/kaggle/working/results') if Path('/kaggle').is_dir() else REPO_ROOT / 'results'
LOG_DIR = Path(os.getenv('CEPHALO_LOG_DIR', str(RESULTS_DIR)))
CHECKPOINT_DIR = Path(os.getenv('CEPHALO_CHECKPOINT_DIR', '/kaggle/working/checkpoints' if Path('/kaggle').is_dir() else str(REPO_ROOT / 'checkpoints')))
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

if Path('/kaggle').is_dir():
    FT_IMAGES = resolve_data_dir(FT_ROOT, Path('images_v2') / 'images_v2')
    FT_ANNOTATIONS = resolve_data_dir(FT_ROOT, Path('annotations_v2') / 'annotations_v2')
else:
    FT_IMAGES = str(Path(FT_ROOT) / 'images')
    FT_ANNOTATIONS = str(Path(FT_ROOT) / 'annotations')

ranking_cfg = make_stage3_finetune_config(
    ft_root=FT_ROOT, pretrained_weights=STAGE2_WEIGHTS,
    log_dir=str(LOG_DIR), checkpoint_dir=str(CHECKPOINT_DIR),
    backbone_weights=None,
)
set_global_seed(ranking_cfg.seed)  # before ranking model/data construction
assert Path(STAGE2_WEIGHTS).name == 'best_model_stage2_v2.weights.h5'
print('Repository:', REPO_ROOT)
print('In-house images:', FT_IMAGES)
print('Stage 2 v2 checkpoint:', STAGE2_WEIGHTS)


## 1. Pre-training audit: rank all 80 images with Stage 2

Each image is evaluated individually (`batch_size=1`), without augmentation and with `training=False`. Rank 1 is worst by 19-landmark image MRE; ties use image ID for deterministic ordering. The table also records maximum landmark error and image-level SDR.


In [ ]:
from evaluate import load_compatible_weights, rank_inhouse_images
from model import build_model

ranking_model, _ = build_model(ranking_cfg)
load_compatible_weights(ranking_model, STAGE2_WEIGHTS, ranking_cfg)  # strict, ranking model only
RANKING_CSV = RESULTS_DIR / 'stage2_all80_image_ranking.csv'
ranking_table = rank_inhouse_images(
    ranking_model, ranking_cfg, image_ids=range(1, 81),
    root=FT_ROOT, image_dir=FT_IMAGES, annotation_dir=FT_ANNOTATIONS,
    csv_path=RANKING_CSV,
)
display(ranking_table)
print('Saved:', RANKING_CSV)
del ranking_model  # training below builds a fresh model; no duplicate load into it


## 2. Manual split and experiment configuration

Edit the next cell after inspecting the ranking. The provided constants remain convenient references, not hard-wired choices. For `n10`, provide ten IDs nested within `REFERENCE_TRAIN20`; the other ten IDs outside the fixed 60 validation set are explicitly reported as unused. Use `head_logits` for the more conservative 10-image scope if desired.


In [ ]:
from config import (
    STAGE3_TRAIN_IDS, STAGE3_VAL_IDS, make_stage3_experiment_config,
    validate_stage3_split,
)

# USER-CONTROLLED VALUES — edit these lists/name/scope.
TRAIN_IMAGE_IDS = list(STAGE3_TRAIN_IDS)  # enter 10 or 20 unique IDs
VAL_IMAGE_IDS = list(STAGE3_VAL_IDS)      # enter the fixed 60 unique IDs
EXPERIMENT_NAME = 'stage3_n20'
TRAINABLE_SCOPE = 'dec2_head_logits'      # optional: 'head_logits'
REFERENCE_TRAIN20 = list(STAGE3_TRAIN_IDS)  # enforces nesting for an n10 experiment

split = validate_stage3_split(
    TRAIN_IMAGE_IDS, VAL_IMAGE_IDS,
    reference_train20=REFERENCE_TRAIN20 if len(TRAIN_IMAGE_IDS) == 10 else None,
)
cfg = make_stage3_experiment_config(
    TRAIN_IMAGE_IDS, VAL_IMAGE_IDS,
    experiment_name=EXPERIMENT_NAME,
    trainable_scope=TRAINABLE_SCOPE,
    reference_train20=REFERENCE_TRAIN20 if len(TRAIN_IMAGE_IDS) == 10 else None,
    ft_root=FT_ROOT, pretrained_weights=STAGE2_WEIGHTS,
    log_dir=str(LOG_DIR), checkpoint_dir=str(CHECKPOINT_DIR),
    backbone_weights=None,
)
set_global_seed(cfg.seed)  # before final loader/model construction
print('Experiment:', cfg.experiment_name, '| training n =', split['sample_count'])
print('Train IDs:', split['train_ids'])
print('Validation IDs:', split['val_ids'])
print('Unused IDs:', split['unused_ids'])
print('Scope:', cfg.trainable_scope)
print('Best-loss checkpoint:', CHECKPOINT_DIR / cfg.output_filename)
print('Best-MRE checkpoint:', CHECKPOINT_DIR / cfg.best_mre_output_filename)

assert cfg.epochs == 25 and cfg.learning_rate == 5e-6 and cfg.weight_decay == 5e-5
assert cfg.decoder_dropout_rate == 0.20 and cfg.checkpoint_monitor == 'val_loss'
assert cfg.lr_scheduler_monitor == cfg.early_stopping_monitor == 'val_loss'
assert cfg.lr_plateau_patience == 2 and cfg.early_stopping_patience == 5
assert cfg.lr_plateau_min_delta == cfg.early_stopping_min_delta == 2e-4
assert cfg.encoder_mode == 'stage1' and cfg.freeze_encoder_batch_norm


## 3. Separate loaders and fresh training model

`train_loader` provides augmented online metrics. `train_eval_loader` evaluates the same IDs without augmentation, without shuffling, and in inference mode after each epoch. Validation is also unaugmented. The fresh training model is **not** manually loaded here; `train()` owns exactly one strict Stage 2 load.


In [ ]:
import tensorflow as tf
from data import InHouseDataset
from model import build_model
from train import train

train_loader = InHouseDataset(
    FT_ROOT, TRAIN_IMAGE_IDS, cfg, training=True, shuffle=True,
    image_dir=FT_IMAGES, annotation_dir=FT_ANNOTATIONS,
)
train_eval_loader = InHouseDataset(
    FT_ROOT, TRAIN_IMAGE_IDS, cfg, training=False, shuffle=False,
    image_dir=FT_IMAGES, annotation_dir=FT_ANNOTATIONS,
)
val_loader = InHouseDataset(
    FT_ROOT, VAL_IMAGE_IDS, cfg, training=False, shuffle=False,
    image_dir=FT_IMAGES, annotation_dir=FT_ANNOTATIONS,
)
model, encoder_layer_names = build_model(cfg)
# No load_weights call: train() performs the training model's sole strict load.
print('Complete batches — online:', len(train_loader), 'train eval:', len(train_eval_loader), 'validation:', len(val_loader))
print('Model outputs:', model.output_shape)
print('Dropout layers have no weights:', {
    layer.name: len(layer.weights) for layer in model.layers
    if isinstance(layer, tf.keras.layers.Dropout)
})


In [ ]:
history = train(
    cfg,
    train_loader=train_loader,
    train_eval_loader=train_eval_loader,
    val_loader=val_loader,
    model=model,
    encoder_layer_names=encoder_layer_names,
)
assert history['incoming_checkpoint'] == cfg.pretrained_weights
print('Parameter counts:', history['parameter_counts'])
print('Best validation loss:', history['best_loss_value'], 'at epoch', history['best_loss_epoch'])
print('Best validation MRE:', history['best_mre_value'], 'at epoch', history['best_mre_epoch'])
print('Stop:', history['stop_reason'], 'at epoch', history['stopped_epoch'])
print('Restored best-loss evaluation:', history['restored_final_validation'])
print('Checkpoints:', history['checkpoint_paths'])
print('History:', history['history_path'])


## 4. Honest learning curves

Raw validation loss is never replaced or smoothed. Its running minimum is shown only as a separately labelled selection trace. Epoch zero is the loaded Stage 2 baseline. Unaugmented inference-mode train evaluation is the primary train/validation comparison; augmented online loss is shown separately.


In [ ]:
import matplotlib.pyplot as plt

epochs = [0] + history['epoch']
train_eval_loss = [history['train_eval_baseline']['loss']] + history['train_eval_loss']
val_loss = [history['baseline']['loss']] + history['val_loss']
running_best = [history['baseline']['loss']] + history['val_loss_running_best']

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].plot(epochs, train_eval_loss, marker='o', label='train eval loss (unaugmented, inference mode)')
axes[0].plot(epochs, val_loss, marker='o', label='raw validation loss')
axes[0].plot(epochs, running_best, linestyle='--', label='running-best validation loss')
axes[0].scatter([0], [val_loss[0]], color='black', zorder=5, label='epoch 0: Stage 2')
axes[0].set(xlabel='Epoch', ylabel='Composite loss', title='Inference-mode train and validation loss')
axes[0].legend(); axes[0].grid(alpha=0.25)

axes[1].plot(history['epoch'], history['train_online_loss'], label='online augmented training loss')
axes[1].plot(history['epoch'], history['train_eval_loss'], label='unaugmented train eval loss')
axes[1].set(xlabel='Epoch', ylabel='Composite loss', title='Online versus inference-mode training loss')
axes[1].legend(); axes[1].grid(alpha=0.25)
plt.tight_layout(); plt.show()


## 5. Primary restored result and optional best-MRE comparison

The in-memory `model` is restored to the **lowest-validation-loss** checkpoint. Subsequent per-landmark analysis should use it. A separately constructed model can strictly load the best-MRE checkpoint for a clearly labelled comparison; never conflate the two outcomes.


In [ ]:
from train import evaluate
from evaluate import load_compatible_weights

primary_metrics = evaluate(model, val_loader, cfg)
print('PRIMARY restored best-loss result')
print('MRE:', primary_metrics['mre'], '| SDR:', primary_metrics['sdr'])
print('Per-landmark MRE:', primary_metrics['per_landmark_mre'])

best_mre_model, _ = build_model(cfg)
load_compatible_weights(best_mre_model, history['checkpoint_paths']['best_mre'], cfg)
best_mre_metrics = evaluate(best_mre_model, val_loader, cfg)
print()
print('SEPARATE best-MRE checkpoint result')
print('MRE:', best_mre_metrics['mre'], '| SDR:', best_mre_metrics['sdr'])


### Interpretation boundary

These are validation/model-selection results. Ranking all 80 images before choosing the split, repeatedly monitoring the 60 validation images, and selecting checkpoints from them means this workflow does not provide an independent test estimate. Preserve the exact split IDs, unused IDs, ranking CSV, history JSON, and both checkpoints with any reported result.


## 6. Comprehensive validation results for the fixed n20 experiment

The following cells use the already restored **primary best-validation-loss model** in `model`. They do not retrain or change the 20-image experiment. They collect one unaugmented, inference-mode prediction per validation image, retain the predicted heatmaps, calculate image- and landmark-level errors, and export paper-ready tables and figures.

In [ ]:
# Collect raw images, raw annotations, predictions, heatmaps, and paper-scale errors.
# This is inference only: no optimizer, augmentation, or checkpoint selection is used here.
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from data import preprocess, read_points, read_rgb

LANDMARK_NAMES = [
    cfg.anatomical_landmarks.get(str(i), f'Landmark {i + 1}')
    for i in range(cfg.num_landmarks)
]
REFERENCE_SCALE = np.array([cfg.original_width, cfg.original_height], dtype=np.float32)
RESULTS_STEM = f'stage3_v3_{EXPERIMENT_NAME}'


def _image_and_annotation_paths(image_id):
    image_path = Path(FT_IMAGES) / f'({int(image_id)}).tif'
    annotation_path = Path(FT_ANNOTATIONS) / f'({int(image_id)}).txt'
    if not image_path.exists():
        matches = sorted(Path(FT_IMAGES).glob(f'*{int(image_id)}*'))
        if not matches:
            raise FileNotFoundError(f'No image found for image ID {image_id}')
        image_path = matches[0]
    if not annotation_path.exists():
        matches = sorted(Path(FT_ANNOTATIONS).glob(f'*{int(image_id)}*'))
        if not matches:
            raise FileNotFoundError(f'No annotation found for image ID {image_id}')
        annotation_path = matches[0]
    return image_path, annotation_path


records = []
for image_id in sorted(VAL_IMAGE_IDS):
    image_path, annotation_path = _image_and_annotation_paths(image_id)
    raw_image = read_rgb(image_path)
    raw_truth = read_points(annotation_path, cfg)
    model_image, truth_norm = preprocess(raw_image, raw_truth, cfg, augmentation=None)
    outputs = model(np.expand_dims(model_image, axis=0), training=False)
    pred_norm = np.asarray(outputs['coords'])[0].astype(np.float32)
    heatmaps = np.asarray(outputs['heatmaps'])[0].astype(np.float32)

    pred_reference = pred_norm * REFERENCE_SCALE
    truth_reference = truth_norm * REFERENCE_SCALE
    error_mm = np.linalg.norm(pred_reference - truth_reference, axis=-1) * cfg.image_resolution
    raw_scale = np.array([raw_image.shape[1], raw_image.shape[0]], dtype=np.float32)
    pred_raw = pred_norm * raw_scale

    records.append({
        'image_id': int(image_id),
        'image_path': str(image_path),
        'annotation_path': str(annotation_path),
        'image': raw_image,
        'truth_raw': raw_truth,
        'truth_norm': truth_norm,
        'pred_norm': pred_norm,
        'pred_raw': pred_raw,
        'heatmaps': heatmaps,
        'error_mm': error_mm,
        'mre_mm': float(np.mean(error_mm)),
        'median_landmark_error_mm': float(np.median(error_mm)),
        'std_landmark_error_mm': float(np.std(error_mm)),
        'max_landmark_error_mm': float(np.max(error_mm)),
        'worst_landmark_index': int(np.argmax(error_mm)),
    })

records_by_id = {record['image_id']: record for record in records}
records_sorted_best = sorted(records, key=lambda item: (item['mre_mm'], item['image_id']))
example_records = {
    'best': records_sorted_best[0],
    'median': records_sorted_best[len(records_sorted_best) // 2],
    'worst': records_sorted_best[-1],
}

thresholds = [float(value) for value in cfg.precision_thresholds]
per_image_rows = []
for record in records:
    row = {
        'image_id': record['image_id'],
        'mre_mm': record['mre_mm'],
        'median_landmark_error_mm': record['median_landmark_error_mm'],
        'std_landmark_error_mm': record['std_landmark_error_mm'],
        'max_landmark_error_mm': record['max_landmark_error_mm'],
        'worst_landmark_index': record['worst_landmark_index'] + 1,
        'worst_landmark': LANDMARK_NAMES[record['worst_landmark_index']],
    }
    for threshold in thresholds:
        label = str(threshold).replace('.', '_')
        row[f'sdr_{label}mm_percent'] = float(np.mean(record['error_mm'] < threshold) * 100.0)
    per_image_rows.append(row)

per_image_table = pd.DataFrame(per_image_rows).sort_values(
    ['mre_mm', 'image_id'], ascending=[False, True], kind='mergesort'
).reset_index(drop=True)
per_image_table.insert(0, 'rank_worst_to_best', np.arange(1, len(per_image_table) + 1))

landmark_rows = []
error_matrix = np.stack([record['error_mm'] for record in records], axis=0)
for landmark_index, landmark_name in enumerate(LANDMARK_NAMES):
    values = error_matrix[:, landmark_index]
    row = {
        'landmark_index': landmark_index + 1,
        'landmark': landmark_name,
        'mean_error_mm': float(np.mean(values)),
        'median_error_mm': float(np.median(values)),
        'std_error_mm': float(np.std(values)),
        'p25_error_mm': float(np.percentile(values, 25)),
        'p75_error_mm': float(np.percentile(values, 75)),
        'max_error_mm': float(np.max(values)),
    }
    for threshold in thresholds:
        label = str(threshold).replace('.', '_')
        row[f'sdr_{label}mm_percent'] = float(np.mean(values < threshold) * 100.0)
    landmark_rows.append(row)
landmark_table = pd.DataFrame(landmark_rows)

selected_landmark_table = pd.DataFrame([
    {
        'example': label,
        'image_id': record['image_id'],
        'landmark_index': index + 1,
        'landmark': LANDMARK_NAMES[index],
        'error_mm': float(record['error_mm'][index]),
    }
    for label, record in example_records.items()
    for index in range(cfg.num_landmarks)
])

RESULTS_DIR = Path(RESULTS_DIR)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
per_image_csv = RESULTS_DIR / f'{RESULTS_STEM}_validation_per_image_metrics.csv'
landmark_csv = RESULTS_DIR / f'{RESULTS_STEM}_validation_landmark_metrics.csv'
selected_csv = RESULTS_DIR / f'{RESULTS_STEM}_selected_examples_landmark_errors.csv'
per_image_table.to_csv(per_image_csv, index=False)
landmark_table.to_csv(landmark_csv, index=False)
selected_landmark_table.to_csv(selected_csv, index=False)

print(f'Collected {len(records)} validation images using the restored primary model.')
print('Best / median / worst image IDs:', {
    label: record['image_id'] for label, record in example_records.items()
})
print('Saved:', per_image_csv)
print('Saved:', landmark_csv)
print('Saved:', selected_csv)
display(per_image_table.head(10))
display(landmark_table)


### 6.1 Aggregate, image-level, and landmark-level error summaries

The tables below separate three useful reporting levels: overall validation performance, variation between radiographs, and variation between landmarks. The image-level table is sorted from worst to best so difficult cases can be inspected directly.

In [ ]:
# Print a compact Results-section summary and save a machine-readable JSON record.
mean_image_mre = float(np.mean([record['mre_mm'] for record in records]))
median_image_mre = float(np.median([record['mre_mm'] for record in records]))
std_image_mre = float(np.std([record['mre_mm'] for record in records]))
summary = {
    'experiment_name': str(EXPERIMENT_NAME),
    'training_image_count': int(len(TRAIN_IMAGE_IDS)),
    'validation_image_count': int(len(VAL_IMAGE_IDS)),
    'trainable_scope': str(cfg.trainable_scope),
    'primary_checkpoint': str(history['checkpoint_paths']['best_loss']),
    'best_loss_epoch': int(history['best_loss_epoch']),
    'best_mre_epoch': int(history['best_mre_epoch']),
    'overall_mre_mm': float(primary_metrics['mre']),
    'overall_sdr_percent': {str(float(k)): float(v) for k, v in primary_metrics['sdr'].items()},
    'image_mre_mean_mm': mean_image_mre,
    'image_mre_median_mm': median_image_mre,
    'image_mre_std_mm': std_image_mre,
    'image_mre_min_mm': float(np.min([record['mre_mm'] for record in records])),
    'image_mre_max_mm': float(np.max([record['mre_mm'] for record in records])),
    'best_image_id': int(example_records['best']['image_id']),
    'median_image_id': int(example_records['median']['image_id']),
    'worst_image_id': int(example_records['worst']['image_id']),
    'landmark_names': LANDMARK_NAMES,
    'per_landmark_mean_mre_mm': [float(value) for value in landmark_table['mean_error_mm']],
}
summary_json = RESULTS_DIR / f'{RESULTS_STEM}_results_summary.json'
summary_json.write_text(json.dumps(summary, indent=2), encoding='utf-8')

print('PRIMARY best-loss validation summary')
print(f"Overall MRE: {summary['overall_mre_mm']:.3f} mm")
print(f"Image-level MRE: mean {mean_image_mre:.3f}, median {median_image_mre:.3f}, SD {std_image_mre:.3f} mm")
print('SDR:', {f'@{key} mm': f'{value:.1f}%' for key, value in summary['overall_sdr_percent'].items()})
print('Best / median / worst:', {
    label: f"ID {record['image_id']} ({record['mre_mm']:.3f} mm)"
    for label, record in example_records.items()
})
print('Most difficult landmarks by mean error:')
display(landmark_table.sort_values('mean_error_mm', ascending=False).head(10))
print('Saved:', summary_json)


### 6.2 Error distributions and landmark-wise performance

These figures are intended for the paper’s quantitative Results section. The boxplot shows the full validation distribution for each landmark; the bar plot shows mean error with standard-deviation whiskers. The final panel shows the complete image-by-landmark error matrix and helps distinguish image-specific failures from consistently difficult landmarks.

In [ ]:
# Landmark-wise distributions and the image-by-landmark error matrix.
fig, axes = plt.subplots(1, 3, figsize=(23, 6), gridspec_kw={'width_ratios': [1.3, 1.4, 1.5]})

axes[0].boxplot(
    [error_matrix[:, index] for index in range(cfg.num_landmarks)],
    labels=[str(index + 1) for index in range(cfg.num_landmarks)],
    showfliers=False,
)
axes[0].axhline(2.0, color='tab:orange', linestyle='--', linewidth=1, label='2 mm')
axes[0].axhline(3.0, color='tab:red', linestyle=':', linewidth=1, label='3 mm')
axes[0].set_title('Landmark error distributions')
axes[0].set_xlabel('Landmark index')
axes[0].set_ylabel('Radial error (mm)')
axes[0].legend(fontsize=8)
axes[0].grid(axis='y', alpha=0.25)

means = landmark_table['mean_error_mm'].to_numpy()
stds = landmark_table['std_error_mm'].to_numpy()
axes[1].bar(np.arange(cfg.num_landmarks), means, yerr=stds, capsize=3, color='steelblue', alpha=0.85)
axes[1].axhline(2.0, color='tab:orange', linestyle='--', linewidth=1)
axes[1].axhline(3.0, color='tab:red', linestyle=':', linewidth=1)
axes[1].set_xticks(np.arange(cfg.num_landmarks), [str(index + 1) for index in range(cfg.num_landmarks)], rotation=0)
axes[1].set_title('Mean landmark error ± SD')
axes[1].set_xlabel('Landmark index')
axes[1].set_ylabel('Mean radial error (mm)')
axes[1].grid(axis='y', alpha=0.25)

order = np.argsort([record['mre_mm'] for record in records])[::-1]
im = axes[2].imshow(error_matrix[order], aspect='auto', cmap='magma', vmin=0)
axes[2].set_xticks(np.arange(cfg.num_landmarks), [str(index + 1) for index in range(cfg.num_landmarks)], rotation=90)
axes[2].set_yticks(np.arange(len(records)), [str(records[index]['image_id']) for index in order])
axes[2].set_title('Validation error matrix (mm)')
axes[2].set_xlabel('Landmark index')
axes[2].set_ylabel('Image ID (worst to best)')
fig.colorbar(im, ax=axes[2], label='Error (mm)', shrink=0.85)
fig.suptitle(f'{EXPERIMENT_NAME}: landmark- and image-level validation errors', y=1.02)
fig.tight_layout()
landmark_figure = RESULTS_DIR / f'{RESULTS_STEM}_landmark_error_distributions'
fig.savefig(landmark_figure.with_suffix('.png'), dpi=220, bbox_inches='tight')
fig.savefig(landmark_figure.with_suffix('.pdf'), bbox_inches='tight')
plt.show()
print('Saved:', landmark_figure.with_suffix('.png'))
print('Saved:', landmark_figure.with_suffix('.pdf'))


### 6.3 Best, median, and worst radiographs with landmark overlays

Cyan markers are ground-truth landmarks, magenta markers are predictions, and yellow segments connect each prediction to its corresponding ground truth. Labels are landmark indices; the accompanying table gives the anatomical names and errors for the three selected examples.

In [ ]:
def _plot_prediction_overlay(axis, record, title):
    image = record['image']
    truth = record['truth_raw']
    prediction = record['pred_raw']
    axis.imshow(image, cmap='gray')
    for index, (truth_point, prediction_point) in enumerate(zip(truth, prediction)):
        axis.plot(
            [truth_point[0], prediction_point[0]],
            [truth_point[1], prediction_point[1]],
            color='yellow', linewidth=0.8, alpha=0.65,
        )
    axis.scatter(truth[:, 0], truth[:, 1], s=25, c='cyan', edgecolors='black', linewidths=0.35, label='Ground truth')
    axis.scatter(prediction[:, 0], prediction[:, 1], s=25, c='magenta', marker='x', linewidths=1.1, label='Prediction')
    for index, (x, y) in enumerate(truth):
        axis.text(x + 8, y - 8, str(index + 1), color='white', fontsize=7, weight='bold',
                  bbox={'facecolor': 'black', 'alpha': 0.45, 'pad': 0.5, 'linewidth': 0})
    axis.set_title(f"{title}: image {record['image_id']} | MRE {record['mre_mm']:.2f} mm")
    axis.axis('off')

fig, axes = plt.subplots(1, 3, figsize=(18, 8))
for axis, (label, record) in zip(axes, example_records.items()):
    _plot_prediction_overlay(axis, record, label.capitalize())
axes[0].legend(loc='lower left', fontsize=8, framealpha=0.85)
fig.suptitle(f'{EXPERIMENT_NAME}: representative validation predictions', y=0.98)
fig.tight_layout()
overlay_figure = RESULTS_DIR / f'{RESULTS_STEM}_best_median_worst_overlays'
fig.savefig(overlay_figure.with_suffix('.png'), dpi=240, bbox_inches='tight')
fig.savefig(overlay_figure.with_suffix('.pdf'), bbox_inches='tight')
plt.show()

selected_display = selected_landmark_table.copy()
selected_display['error_mm'] = selected_display['error_mm'].round(3)
display(selected_display)
print('Saved:', overlay_figure.with_suffix('.png'))
print('Saved:', overlay_figure.with_suffix('.pdf'))


### 6.4 Predicted heatmaps and spatial localization

The first panel overlays the maximum predicted landmark probability over the median-error radiograph. The 19-panel grid shows the individual spatial-softmax heatmap for every landmark; cyan is the ground truth and magenta is the decoded prediction in heatmap coordinates. This provides a qualitative check of localization sharpness and failure modes beyond the coordinate metrics.

In [ ]:
heatmap_record = example_records['median']
heatmaps = heatmap_record['heatmaps']
heatmap_height, heatmap_width = heatmaps.shape[:2]
truth_hm = heatmap_record['truth_norm'] * np.array([heatmap_width - 1, heatmap_height - 1], dtype=np.float32)
pred_hm = heatmap_record['pred_norm'] * np.array([heatmap_width - 1, heatmap_height - 1], dtype=np.float32)

fig = plt.figure(figsize=(20, 17), constrained_layout=True)
grid = fig.add_gridspec(5, 5, height_ratios=[1.15, 1, 1, 1, 1])
ax_overlay = fig.add_subplot(grid[0, :])
ax_overlay.imshow(heatmap_record['image'], cmap='gray')
max_projection = np.max(heatmaps, axis=-1)
ax_overlay.imshow(max_projection, cmap='turbo', alpha=0.35, extent=(0, heatmap_record['image'].shape[1], heatmap_record['image'].shape[0], 0))
ax_overlay.scatter(heatmap_record['truth_raw'][:, 0], heatmap_record['truth_raw'][:, 1], c='cyan', s=22, label='Ground truth')
ax_overlay.scatter(heatmap_record['pred_raw'][:, 0], heatmap_record['pred_raw'][:, 1], c='magenta', marker='x', s=28, label='Prediction')
ax_overlay.set_title(f"Median-error image {heatmap_record['image_id']}: maximum predicted heatmap projection")
ax_overlay.axis('off')
ax_overlay.legend(loc='lower left', fontsize=8)

heatmap_axes = []
for index in range(cfg.num_landmarks):
    axis = fig.add_subplot(grid[1 + index // 5, index % 5])
    heatmap_axes.append(axis)
    image = axis.imshow(heatmaps[:, :, index], cmap='magma', vmin=0, vmax=float(np.max(heatmaps)))
    axis.scatter(truth_hm[index, 0], truth_hm[index, 1], c='cyan', s=18, edgecolors='black', linewidths=0.3)
    axis.scatter(pred_hm[index, 0], pred_hm[index, 1], c='magenta', marker='x', s=22, linewidths=1)
    axis.set_title(f'{index + 1}: {LANDMARK_NAMES[index]}', fontsize=8)
    axis.set_xticks([])
    axis.set_yticks([])
fig.colorbar(image, ax=heatmap_axes, shrink=0.65, label='Spatial probability')
heatmap_figure = RESULTS_DIR / f'{RESULTS_STEM}_median_heatmaps'
fig.savefig(heatmap_figure.with_suffix('.png'), dpi=220, bbox_inches='tight')
fig.savefig(heatmap_figure.with_suffix('.pdf'), bbox_inches='tight')
plt.show()
print('Saved:', heatmap_figure.with_suffix('.png'))
print('Saved:', heatmap_figure.with_suffix('.pdf'))


### 6.5 Paper-ready reporting block

Use the exported CSV, JSON, PNG, and PDF artifacts together with the fixed experiment metadata. The quantitative claims should be based on the primary restored best-validation-loss checkpoint; the images and heatmaps are qualitative diagnostics for the same checkpoint.

In [ ]:
# A final compact block that can be copied into a Results/Discussion draft.
worst_landmark_row = landmark_table.iloc[int(np.argmax(landmark_table['mean_error_mm']))]
print(
    f"Stage 3 ({EXPERIMENT_NAME}) adapted the model using {len(TRAIN_IMAGE_IDS)} training images "
    f"and evaluated {len(VAL_IMAGE_IDS)} validation images. The primary restored checkpoint "
    f"achieved an overall MRE of {primary_metrics['mre']:.3f} mm. Image-level MRE was "
    f"{mean_image_mre:.3f} ± {std_image_mre:.3f} mm (median {median_image_mre:.3f} mm). "
    f"The most difficult landmark by mean radial error was landmark "
    f"{int(worst_landmark_row['landmark_index'])} ({worst_landmark_row['landmark']}), "
    f"at {worst_landmark_row['mean_error_mm']:.3f} mm."
)
print('Artifacts are in:', RESULTS_DIR)
print('  -', per_image_csv.name)
print('  -', landmark_csv.name)
print('  -', selected_csv.name)
print('  -', summary_json.name)
